# OpenAI Agents SDK + Ollama demo notebook

This notebook demonstrates **three ways to solve the same business workflow** with the Python **OpenAI Agents SDK** running against **Ollama** with the **`qwen3.5:9b`** model:

1. **Single-agent flow** — one agent owns the whole task
2. **Multi-agent handoff flow** — specialist agents hand work off to the next specialist
3. **Multi-agent orchestrator flow** — an orchestrator agent uses specialist agents **as tools**

## Shared business issue

All three flows solve the same task:

> Review an order request, verify stock, compute pricing, estimate shipping, and produce:
> - an **operations summary**
> - a **customer-facing message**

## Why this notebook uses the Chat Completions model path

Ollama exposes an OpenAI-compatible API, and the Agents SDK supports non-OpenAI providers by pointing an `AsyncOpenAI` client at a compatible `base_url`. For local/provider-compatible setups, the most pragmatic path is often `OpenAIChatCompletionsModel`.

## Before you run

Make sure Ollama is installed and the model is available locally:

```bash
ollama pull qwen3.5:9b
ollama serve
```

By default this notebook expects Ollama at:

```text
http://localhost:11434/v1
```

In [ ]:
# If needed, install the exact SDK line used in this notebook.
# %pip install -q "openai-agents==0.14.4" "openai>=2.0.0"

In [1]:
import json
import os

from agents import (
    Agent,
    ModelSettings,
    OpenAIChatCompletionsModel,
    Runner,
    function_tool,
    set_tracing_disabled,
)
from openai import AsyncOpenAI

# Tracing uploads to OpenAI by default. Disable it for local/Ollama runs unless
# you have explicitly configured a tracing processor or export API key.
set_tracing_disabled(True)

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11500/v1")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen3.5:9b")
OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY", "ollama")  # ignored locally by Ollama

client = AsyncOpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key=OLLAMA_API_KEY,
)


def build_model():
    return OpenAIChatCompletionsModel(
        model=OLLAMA_MODEL,
        openai_client=client,
    )


print("Using:")
print("  base_url:", OLLAMA_BASE_URL)
print("  model:   ", OLLAMA_MODEL)

Using:
  base_url: http://localhost:11500/v1
  model:    qwen3.5:9b


In [3]:
# Optional sanity check: verify that the OpenAI-compatible endpoint is reachable.
models = await client.models.list()
available_models = [m.id for m in models.data]
print("First models exposed by Ollama /v1/models:")
print(available_models[:10])

if OLLAMA_MODEL not in available_models:
    print(f"\nWarning: {OLLAMA_MODEL!r} was not listed. Double-check the exact model tag in Ollama.")

First models exposed by Ollama /v1/models:
['qwen3.5:9b']


## Shared demo data

The tools below operate over an in-memory mini back office:

- customer profile data
- inventory data
- shipping rules

This keeps the example deterministic and focused on **agent orchestration**, not external APIs.

In [4]:
CUSTOMERS = {
    "CUST-003": {
        "name": "Marta Ruiz",
        "loyalty_tier": "gold",
        "region": "Madrid",
        "preferred_shipping": "express",
    },
    "CUST-009": {
        "name": "Leo Navarro",
        "loyalty_tier": "standard",
        "region": "Barcelona",
        "preferred_shipping": "standard",
    },
}

CATALOG = {
    "wireless_mouse": {"unit_price": 25.0, "in_stock": 8, "weight_kg": 0.2},
    "usb_c_hub": {"unit_price": 45.0, "in_stock": 3, "weight_kg": 0.15},
    "laptop_stand": {"unit_price": 60.0, "in_stock": 5, "weight_kg": 1.1},
    "mechanical_keyboard": {"unit_price": 95.0, "in_stock": 1, "weight_kg": 0.9},
}

LOYALTY_DISCOUNTS = {
    "standard": 0.00,
    "silver": 0.05,
    "gold": 0.10,
    "platinum": 0.15,
}

COUPONS = {
    "SPRING10": 0.10,
    "SAVE5": 0.05,
    None: 0.00,
    "": 0.00,
}

SHIPPING_RULES = {
    "standard": {"days": 5, "base_cost": 7.5},
    "express": {"days": 2, "base_cost": 14.0},
    "same_day": {"days": 0, "base_cost": 30.0},
}

In [5]:
@function_tool
def get_customer_profile(customer_id: str) -> str:
    """Return the customer profile as JSON for a given customer ID."""
    customer = CUSTOMERS.get(customer_id)
    if not customer:
        return json.dumps({"found": False, "customer_id": customer_id}, indent=2)
    return json.dumps({"found": True, "customer_id": customer_id, **customer}, indent=2)


@function_tool
def check_inventory(order_json: str) -> str:
    """
    Check inventory for an order.

    Expects order_json with this shape:
    {
      "items": [
        {"sku": "wireless_mouse", "qty": 3}
      ]
    }
    """
    payload = json.loads(order_json)
    results = []

    for item in payload["items"]:
        sku = item["sku"]
        qty = int(item["qty"])
        product = CATALOG.get(sku)
        if not product:
            results.append(
                {
                    "sku": sku,
                    "requested_qty": qty,
                    "available": False,
                    "reason": "unknown_sku",
                }
            )
            continue

        in_stock = product["in_stock"]
        results.append(
            {
                "sku": sku,
                "requested_qty": qty,
                "in_stock_qty": in_stock,
                "available": in_stock >= qty,
                "backorder_qty": max(0, qty - in_stock),
            }
        )

    return json.dumps({"items": results}, indent=2)


@function_tool
def price_order(order_json: str) -> str:
    """
    Compute order pricing.

    Expects:
    {
      "customer_id": "CUST-003",
      "coupon_code": "SPRING10",
      "items": [{"sku": "wireless_mouse", "qty": 1}]
    }
    """
    payload = json.loads(order_json)
    customer = CUSTOMERS[payload["customer_id"]]
    loyalty_tier = customer["loyalty_tier"]
    loyalty_discount = LOYALTY_DISCOUNTS.get(loyalty_tier, 0.0)
    coupon_discount = COUPONS.get(payload.get("coupon_code"), 0.0)

    lines = []
    subtotal = 0.0

    for item in payload["items"]:
        sku = item["sku"]
        qty = int(item["qty"])
        product = CATALOG[sku]
        line_total = product["unit_price"] * qty
        subtotal += line_total
        lines.append(
            {
                "sku": sku,
                "qty": qty,
                "unit_price": product["unit_price"],
                "line_total": round(line_total, 2),
            }
        )

    loyalty_amount = round(subtotal * loyalty_discount, 2)
    after_loyalty = subtotal - loyalty_amount
    coupon_amount = round(after_loyalty * coupon_discount, 2)
    discounted_subtotal = round(after_loyalty - coupon_amount, 2)
    tax = round(discounted_subtotal * 0.21, 2)
    grand_total = round(discounted_subtotal + tax, 2)

    result = {
        "customer_id": payload["customer_id"],
        "loyalty_tier": loyalty_tier,
        "coupon_code": payload.get("coupon_code"),
        "lines": lines,
        "subtotal": round(subtotal, 2),
        "loyalty_discount_pct": loyalty_discount,
        "loyalty_discount_amount": loyalty_amount,
        "coupon_discount_pct": coupon_discount,
        "coupon_discount_amount": coupon_amount,
        "discounted_subtotal": discounted_subtotal,
        "tax": tax,
        "grand_total": grand_total,
    }
    return json.dumps(result, indent=2)


@function_tool
def estimate_shipping(order_json: str) -> str:
    """
    Estimate shipping options.

    Expects:
    {
      "postal_code": "28013",
      "shipping_preference": "express",
      "items": [{"sku": "wireless_mouse", "qty": 1}]
    }
    """
    payload = json.loads(order_json)
    items = payload["items"]
    preference = payload.get("shipping_preference", "standard")
    postal_code = str(payload["postal_code"])

    total_weight = 0.0
    for item in items:
        product = CATALOG[item["sku"]]
        total_weight += product["weight_kg"] * int(item["qty"])

    madrid_priority = postal_code.startswith(("28", "29"))
    options = []

    for method, rule in SHIPPING_RULES.items():
        if method == "same_day" and not madrid_priority:
            continue

        surcharge = 0.0
        if total_weight > 2.0:
            surcharge += 3.5
        if method == "same_day":
            surcharge += 4.0

        total_cost = round(rule["base_cost"] + surcharge, 2)
        options.append(
            {
                "method": method,
                "eta_days": rule["days"],
                "estimated_cost": total_cost,
                "recommended": method == preference if preference in SHIPPING_RULES else False,
            }
        )

    best = min(options, key=lambda x: (x["eta_days"], x["estimated_cost"]))
    cheapest = min(options, key=lambda x: x["estimated_cost"])

    return json.dumps(
        {
            "postal_code": postal_code,
            "total_weight_kg": round(total_weight, 2),
            "options": options,
            "fastest_option": best,
            "cheapest_option": cheapest,
        },
        indent=2,
    )

## The shared input: one business task for all flows

In [6]:
CASE = {
    "customer_id": "CUST-003",
    "coupon_code": "SPRING10",
    "postal_code": "28013",
    "items": [
        {"sku": "wireless_mouse", "qty": 3},
        {"sku": "usb_c_hub", "qty": 2},
        {"sku": "laptop_stand", "qty": 1},
    ],
}

CASE_PROMPT = f"""
You are reviewing this order request:

{json.dumps(CASE, indent=2)}

Do all of the following:
1. Verify stock for every item.
2. Compute pricing using the customer profile and coupon code.
3. Estimate shipping options for the postal code.
4. Return:
   - an OPERATIONS SUMMARY
   - a CUSTOMER MESSAGE

Important formatting rules:
- Be explicit about which tools you used.
- Mention stock availability item by item.
- Mention subtotal, discounts, tax, and grand total.
- Recommend one shipping option and explain why.
- Keep the customer message concise and friendly.
""".strip()

print(CASE_PROMPT)

You are reviewing this order request:

{
  "customer_id": "CUST-003",
  "coupon_code": "SPRING10",
  "postal_code": "28013",
  "items": [
    {
      "sku": "wireless_mouse",
      "qty": 3
    },
    {
      "sku": "usb_c_hub",
      "qty": 2
    },
    {
      "sku": "laptop_stand",
      "qty": 1
    }
  ]
}

Do all of the following:
1. Verify stock for every item.
2. Compute pricing using the customer profile and coupon code.
3. Estimate shipping options for the postal code.
4. Return:
   - an OPERATIONS SUMMARY
   - a CUSTOMER MESSAGE

Important formatting rules:
- Be explicit about which tools you used.
- Mention stock availability item by item.
- Mention subtotal, discounts, tax, and grand total.
- Recommend one shipping option and explain why.
- Keep the customer message concise and friendly.


## 1) Single-agent flow

This is the simplest setup:

- one agent
- all business tools available
- one `Runner.run(...)`
- the SDK loops internally until the agent is done

In [12]:
single_agent = Agent(
    name="single_order_agent",
    instructions=(
        "You are a capable operations agent. "
        "Solve the full order-review task yourself. "
        "Use tools when needed. "
        "Do not guess data that can be retrieved from tools."
    ),
    model=build_model(),
    model_settings=ModelSettings(temperature=0.1),
    tools=[
        get_customer_profile,
        check_inventory,
        price_order,
        estimate_shipping,
    ],
)

In [18]:
single_result = await Runner.run(single_agent, CASE_PROMPT, max_turns=12)
print(single_result.final_output)

## OPERATIONS SUMMARY

### Stock Verification (Tool: check_inventory)
All items are in stock and available for fulfillment:

| SKU | Requested Qty | In Stock Qty | Status |
|-----|---------------|--------------|--------|
| wireless_mouse | 3 | 8 | ✅ Available |
| usb_c_hub | 2 | 3 | ✅ Available |
| laptop_stand | 1 | 5 | ✅ Available |

### Pricing Calculation (Tool: price_order)
- **Subtotal:** $225.00
- **Loyalty Discount (10%):** -$22.50
- **Coupon Discount (SPRING10, 10%):** -$20.25
- **Discounted Subtotal:** $182.25
- **Tax:** $38.27
- **Grand Total:** $220.52

### Shipping Options (Tool: estimate_shipping)
| Method | ETA | Cost | Recommended |
|--------|-----|------|-------------|
| Standard | 5 days | $7.50 | No |
| Express | 2 days | $14.00 | ✅ Yes |
| Same Day | 0 days | $34.00 | No |

**Recommended Shipping:** Express ($14.00, 2 days)
**Reason:** Express shipping is recommended as it provides a good balance between speed and cost. It delivers in just 2 days for a reasonable $1

## 2) Multi-agent handoff flow

Here we model a sequential workflow of specialists:

- **Coordinator agent** → starts the workflow
- **Inventory agent** → checks stock
- **Pricing agent** → prices the order
- **Shipping agent** → evaluates delivery options
- **Writer agent** → composes the final result

In this pattern, control is *handed off* from one agent to the next.

In [16]:
writer_agent = Agent(
    name="writer_agent",
    instructions=(
        "You are the final writer in the workflow. "
        "Read the conversation history and produce the final answer. "
        "Your final answer must contain exactly these sections: "
        "1) OPERATIONS SUMMARY, 2) CUSTOMER MESSAGE. "
        "Do not hand off further."
    ),
    handoff_description="Final writer that turns collected workflow outputs into the end result.",
    model=build_model(),
    model_settings=ModelSettings(temperature=0.1),
)

shipping_agent = Agent(
    name="shipping_agent",
    instructions=(
        "You are the shipping specialist. "
        "Use the shipping tool to estimate shipping options for the order. "
        "Summarize your shipping findings briefly in the conversation, then hand off to writer_agent."
        "Always hand off to writer_agent, even if you are tempted to solve parts of the task yourself. "
    ),
    handoff_description="Shipping specialist that estimates delivery options and then hands off to the writer.",
    handoffs=[writer_agent],
    model=build_model(),
    model_settings=ModelSettings(temperature=0.1),
    tools=[estimate_shipping],
)

pricing_agent = Agent(
    name="pricing_agent",
    instructions=(
        "You are the pricing specialist. "
        "Use customer and pricing tools to calculate totals and discounts. "
        "Summarize your pricing findings briefly in the conversation, then hand off to shipping_agent."
        "Always hand off to shipping_agent, even if you are tempted to solve parts of the task yourself. "
    ),
    handoff_description="Pricing specialist that computes totals and discounts before handing off to shipping.",
    handoffs=[shipping_agent],
    model=build_model(),
    model_settings=ModelSettings(temperature=0.1),
    tools=[get_customer_profile, price_order],
)

inventory_agent = Agent(
    name="inventory_agent",
    instructions=(
        "You are the inventory specialist. "
        "Use the inventory tool to verify stock for every order line. "
        "Summarize your stock findings briefly in the conversation, then hand off to pricing_agent."
        "Always hand off to pricing_agent, even if you are tempted to solve parts of the task yourself. "
    ),
    handoff_description="Inventory specialist that verifies stock before handing off to pricing.",
    handoffs=[pricing_agent],
    model=build_model(),
    model_settings=ModelSettings(temperature=0.1),
    tools=[check_inventory],
)

handoff_coordinator = Agent(
    name="handoff_coordinator",
    instructions=(
        "You are the workflow coordinator. "
        "Start by handing this order-review task to inventory_agent. "
        "Do not solve the task yourself."
        "Always hand off to inventory_agent, even if you are tempted to solve parts of the task yourself. "
    ),
    handoff_description="Coordinator that starts the specialist handoff chain.",
    handoffs=[inventory_agent],
    model=build_model(),
    model_settings=ModelSettings(temperature=0.1),
)

In [22]:
handoff_result = await Runner.run(handoff_coordinator, CASE_PROMPT, max_turns=16)
print(handoff_result)

RunResult:
- Last agent: Agent(name="inventory_agent", ...)
- Final output (str):
    ### OPERATIONS SUMMARY
    
    **Stock Verification (Tool: `check_inventory`)**
    - **wireless_mouse**: 3 requested, 8 in stock. ✅ Available.
    - **usb_c_hub**: 2 requested, 3 in stock. ✅ Available.
    - **laptop_stand**: 1 requested, 5 in stock. ✅ Available.
    
    All items are in stock and ready to ship.
    
    **Pricing & Discounts**
    - Subtotal: $129.99
    - Discount (SPRING10): -$13.00
    - Tax: $11.69
    - **Grand Total: $128.68**
    
    **Shipping Recommendation**
    - **Recommended**: Standard Ground (3-5 business days)
    - **Why**: Best balance of cost and speed for this order size.
    
    ---
    
    ### CUSTOMER MESSAGE
    
    Hi there! Your order is all set. We’ve got everything in stock and ready to go. With your SPRING10 coupon, your total comes to $128.68. We recommend Standard Ground shipping for the best value. Thanks for shopping with us! 🛒
- 5 new item(s)


## 3) Multi-agent orchestrator flow (agents as tools)

This is the classic **manager/orchestrator** pattern:

- an **orchestrator agent** owns the user conversation
- specialist agents are exposed as **tools**
- the orchestrator decides which specialist tools to call and how to combine their outputs

### Clear responsibilities

- **Inventory tool-agent**: only stock validation
- **Pricing tool-agent**: only customer lookup + price calculation
- **Shipping tool-agent**: only shipping estimation
- **Orchestrator**: planning, calling the right sub-agents, and composing the final answer

In [20]:
inventory_tool_agent = Agent(
    name="inventory_tool_agent",
    instructions=(
        "You are an inventory specialist. "
        "Your only responsibility is to verify stock using the available tool(s). "
        "Return a concise but complete inventory report."
    ),
    model=build_model(),
    model_settings=ModelSettings(temperature=0.1),
    tools=[check_inventory],
)

pricing_tool_agent = Agent(
    name="pricing_tool_agent",
    instructions=(
        "You are a pricing specialist. "
        "Your only responsibility is to use the customer and pricing tools and return a pricing report."
    ),
    model=build_model(),
    model_settings=ModelSettings(temperature=0.1),
    tools=[get_customer_profile, price_order],
)

shipping_tool_agent = Agent(
    name="shipping_tool_agent",
    instructions=(
        "You are a shipping specialist. "
        "Your only responsibility is to estimate shipping options and return a shipping report."
    ),
    model=build_model(),
    model_settings=ModelSettings(temperature=0.1),
    tools=[estimate_shipping],
)

orchestrator_agent = Agent(
    name="order_orchestrator_agent",
    instructions=(
        "You are the orchestrator for the order-review workflow. "
        "You must use the specialist agent-tools rather than solving specialty tasks yourself. "
        "Call the inventory tool-agent, pricing tool-agent, and shipping tool-agent as needed, "
        "then produce the final answer with exactly these sections: "
        "1) OPERATIONS SUMMARY, 2) CUSTOMER MESSAGE. "
        "Mention which specialist tools you used."
    ),
    model=build_model(),
    model_settings=ModelSettings(temperature=0.1),
    tools=[
        inventory_tool_agent.as_tool(
            tool_name="inventory_review",
            tool_description="Validate stock availability for the order and return an inventory report.",
        ),
        pricing_tool_agent.as_tool(
            tool_name="pricing_review",
            tool_description="Calculate discounts, tax, and total for the order and return a pricing report.",
        ),
        shipping_tool_agent.as_tool(
            tool_name="shipping_review",
            tool_description="Estimate shipping options for the order and return a shipping report.",
        ),
    ],
)

In [21]:
orchestrator_result = await Runner.run(orchestrator_agent, CASE_PROMPT, max_turns=16)
print(orchestrator_result.final_output)

## OPERATIONS SUMMARY

**Specialist Tools Used:** inventory_review, pricing_review, shipping_review

**Stock Availability (Item by Item):**
- **wireless_mouse**: 3 units requested, 8 units in stock → ✅ Available
- **usb_c_hub**: 2 units requested, 3 units in stock → ✅ Available
- **laptop_stand**: 1 unit requested, 5 units in stock → ✅ Available

All items are in stock and ready for fulfillment. No backorders required.

**Pricing Details:**
- **Subtotal**: $225.00
- **Loyalty Discount (Gold Tier, 10%)**: -$22.50
- **Coupon Discount (SPRING10, 10%)**: -$20.25
- **Total Discounts**: -$42.75
- **Discounted Subtotal**: $182.25
- **Tax**: $38.27
- **Grand Total**: $220.52

**Shipping Options:**
- Standard: 5 days, $7.50
- Express: 2 days, $14.00
- Same Day: 0 days, $34.00

**Recommended Shipping Option:** Express ($14.00, 2 days)
**Why Recommended:** Express offers the best balance of speed and cost. It delivers in just 2 days for a reasonable $14.00, compared to Standard at $7.50 (5 days) 

## Side-by-side comparison

In [23]:
comparison = {
    "single_agent": str(single_result.final_output),
    "multi_agent_handoff": str(handoff_result.final_output),
    "multi_agent_orchestrator": str(orchestrator_result.final_output),
}

for name, output in comparison.items():
    print("=" * 100)
    print(name.upper())
    print("=" * 100)
    print(output)
    print()

SINGLE_AGENT
## OPERATIONS SUMMARY

### Stock Verification (Tool: check_inventory)
All items are in stock and available for fulfillment:

| SKU | Requested Qty | In Stock Qty | Status |
|-----|---------------|--------------|--------|
| wireless_mouse | 3 | 8 | ✅ Available |
| usb_c_hub | 2 | 3 | ✅ Available |
| laptop_stand | 1 | 5 | ✅ Available |

### Pricing Calculation (Tool: price_order)
- **Subtotal:** $225.00
- **Loyalty Discount (10%):** -$22.50
- **Coupon Discount (SPRING10, 10%):** -$20.25
- **Discounted Subtotal:** $182.25
- **Tax:** $38.27
- **Grand Total:** $220.52

### Shipping Options (Tool: estimate_shipping)
| Method | ETA | Cost | Recommended |
|--------|-----|------|-------------|
| Standard | 5 days | $7.50 | No |
| Express | 2 days | $14.00 | ✅ Yes |
| Same Day | 0 days | $34.00 | No |

**Recommended Shipping:** Express ($14.00, 2 days)
**Reason:** Express shipping is recommended as it provides a good balance between speed and cost. It delivers in just 2 days for a 

## What each pattern is demonstrating

### Single-agent flow
Use when:
- the task is modest
- one agent can safely own the whole plan
- you want the least orchestration code

### Multi-agent handoff
Use when:
- you want the *active* agent to change over time
- each specialist should inherit the conversation and continue from there
- your workflow is naturally stage-based

### Multi-agent orchestrator
Use when:
- you want one central decision-maker
- specialists should stay bounded and callable
- you want tighter control over responsibilities and composition

## Practical note for Ollama / local models

Local models can vary in how reliably they follow tool-calling conventions. If you see weak handoff or tool behavior:

- lower the task complexity
- tighten agent instructions
- reduce tool surface area
- keep tool inputs/output schemas simple
- increase `max_turns` a bit
- try a model variant that is stronger at tool use

## Optional extension ideas

You can extend this notebook by:

- adding structured output types
- adding retries / validation guardrails
- logging intermediate run items
- measuring token use or latency per pattern
- replacing in-memory tools with real APIs